In [2]:
# setup: load the data and define the variable groups
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv("../data/german_credit.csv")
target = "Creditability"
continuous = ["Duration of Credit (month)", "Credit Amount", "Age (years)"]
numeric = continuous + ["No of Credits at this Bank", "No of dependents"]
ordinal = ["Length of current employment", "Instalment per cent",
           "Duration in Current address", "Occupation"]
nominal = ["Account Balance", "Payment Status of Previous Credit", "Purpose",
           "Value Savings/Stocks", "Sex & Marital Status", "Guarantors",
           "Most valuable available asset", "Concurrent Credits",
           "Type of apartment", "Telephone", "Foreign Worker"]
print(df.shape)

(1000, 21)


In [3]:
# Shapiro-Wilk normality test on the continuous variables
for col in continuous:
    w, p = stats.shapiro(df[col])
    verdict = "NOT normal" if p < 0.05 else "normal"
    print(f"{col}: W = {w:.3f}, p = {p:.2e} -> {verdict}")

Duration of Credit (month): W = 0.900, p = 4.91e-25 -> NOT normal
Credit Amount: W = 0.793, p = 8.62e-34 -> NOT normal
Age (years): W = 0.918, p = 6.92e-23 -> NOT normal


## Normality: Shapiro-Wilk

| Variable | W | p | Result |
|---|---|---|---|
| Credit Amount | 0.793 | < 0.001 | Not normal |
| Duration | 0.900 | < 0.001 | Not normal |
| Age | 0.918 | < 0.001 | Not normal |

- All three continuous variables are not normal (p < 0.001), confirming the right skew seen in Phase 1.
- Credit Amount is the furthest from normal (lowest W), matching its highest skew (1.95).
- Decision: use rank-based tests (Mann-Whitney U, Spearman) instead of the t-test and Pearson.

In [5]:
# Mann-Whitney U: do good and bad customers differ on each continuous variable?
good = df[df[target] == 1]
bad = df[df[target] == 0]

for col in continuous:
    u, p = stats.mannwhitneyu(good[col], bad[col])
    prob = u / (len(good) * len(bad))
    print(f"{col}: median good = {good[col].median()}, median bad = {bad[col].median()}, "
          f"U = {u:.0f}, p = {p:.2e}, P(good > bad) = {prob:.2f}")

Duration of Credit (month): median good = 18.0, median bad = 24.0, U = 77996, p = 7.98e-11, P(good > bad) = 0.37
Credit Amount: median good = 2244.0, median bad = 2574.5, U = 93480, p = 5.92e-03, P(good > bad) = 0.45
Age (years): median good = 34.0, median bad = 31.0, U = 119972, p = 3.45e-04, P(good > bad) = 0.57


## Mann-Whitney U: good vs bad customers

| Variable | Median good | Median bad | p | P(good > bad) |
|---|---|---|---|---|
| Duration | 18 months | 24 months | < 0.001 | 0.37 |
| Credit Amount | 2,244 DM | 2,575 DM | 0.006 | 0.45 |
| Age | 34 years | 31 years | < 0.001 | 0.57 |

- All three differences are statistically significant (p < 0.05).
- Bad customers tend to take longer and larger loans; good customers tend to be older.
- Duration shows the strongest difference (P = 0.37, furthest from 0.50); Credit Amount the weakest.
- P(good > bad) = the chance that a random good customer has a higher value than a random bad one (0.50 = no difference).

In [6]:
# Welch's t-test (compares means) next to Mann-Whitney (compares ranks)
for col in continuous:
    t, p_t = stats.ttest_ind(good[col], bad[col], equal_var=False)
    u, p_u = stats.mannwhitneyu(good[col], bad[col])
    print(f"{col}: mean good = {good[col].mean():.1f}, mean bad = {bad[col].mean():.1f}, "
          f"t = {t:.2f}, p (t-test) = {p_t:.2e}, p (Mann-Whitney) = {p_u:.2e}")

Duration of Credit (month): mean good = 19.2, mean bad = 24.9, t = -6.47, p (t-test) = 2.40e-10, p (Mann-Whitney) = 7.98e-11
Credit Amount: mean good = 2985.4, mean bad = 3938.1, t = -4.26, p (t-test) = 2.48e-05, p (Mann-Whitney) = 5.92e-03
Age (years): mean good = 36.2, mean bad = 34.0, t = 2.91, p (t-test) = 3.78e-03, p (Mann-Whitney) = 3.45e-04
